In [109]:
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


In [110]:
#DATA_PATH = "/home/denklewer/Data/gbm122/pVACseq/"
DATA_PATH = "D:/Google Drive/Work/ITMO/data/schreiber_12_mers/"

In [111]:
from mhcgnomes import parse

# read total data

source_total_df =  pd.read_csv("{}schreiber_total_data_12_mers.csv".format(DATA_PATH),sep=";")
ALLELES = list(set(source_total_df['allele']))
per_allele_mt_df =dict()
per_allele_wt_df = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    df_mt = pd.read_csv("{}schreiber_data_12_mers_{}_mt.csv".format(DATA_PATH,target_allele_name),sep=";")
    df_wt = pd.read_csv("{}schreiber_data_12_mers_{}_wt.csv".format(DATA_PATH,target_allele_name),sep=";")
    
    per_allele_mt_df[allele_name] = df_mt
    per_allele_wt_df[allele_name] = df_wt
    
    assert 'peptide' in df_mt.columns
    assert 'length' in df_mt.columns
    assert RESULT_COLUMNS['id'] in df_mt.columns
    assert 'peptide' in df_wt.columns
    assert 'length' in df_wt.columns
    assert RESULT_COLUMNS['id'] in df_wt.columns
    
    
def defineOrganism(allele_name):
    parse_result = parse(allele_name)
    if parse_result.is_mouse:
        return "Mouse"
    elif parse_result.is_human:
        return "Human"
    else:
        print("organism not found")
        return None


def defineClass(allele_name):
    parse_result = parse(allele_name)
    if parse_result.has_mhc_class:
        if parse_result.is_class1:
            return 'I'
        elif parse_result.is_class2:
            return 'II'
        else:
            print("Not class I or class II???!")
    else:
        print("Result without class")
    return None
ALLELES

['H-2-Dd', 'H-2-Ld', 'H-2-Kd']

# 1. Read all the predictions

In [112]:
import os


In [113]:
per_method_columns = {
    'NetMHC': ['netMHC-3.4_score', 'netMHC-3.4_affinity', 'netMHC-4.0_score',  'netMHC-4.0_affinity','netMHC-4.0_percentile_rank', \
               'netMHCpan-4.1_score_ba', 'netMHCpan-4.1_score_el', 'netMHCpan-4.1_affinity', 'netMHCpan-4.1_percentile_rank_ba', 'netMHCpan-4.1_percentile_rank_el',
                'netMHCIIpan-4.0_score_ba', 'netMHCIIpan-4.0_score_el', 'netMHCIIpan-4.0_affinity', 'netMHCIIpan-4.0_percentile_rank_ba', 'netMHCIIpan-4.0_percentile_rank_el',],
    'netMHC-3.4': [ 'score', 'affinity'],
    'netMHC-4.0':[ 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':[ 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'netMHCIIpan-4.0' :['score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],

    'mhcflurry':['affinity', 'percentile_rank_ba', 'processing_score', 'score_el','percentile_rank_el' ],
    'mhcnuggets':[ 'score', 'affinity', 'percentile_rank'],
    'acme':['score'],
    'puffin':['score', 'epistemic_var', 'aleatoric_var', 'binding_likelihood', 'percentile_rank'],
    'mhcseqnet':['score'],
    'mhcattnnet':[ 'score'],
    'ghmm': ['HMMB', 'percentile_rank']
}



In [114]:
PREDICTION_PATH = "D:/Google Drive/Work/ITMO/MHC-Epitop-Prediction/prediction_results/schreiber_12_mers/"
methods =  next(os.walk(PREDICTION_PATH))[1] # read all the folders in a prediction path
methods.remove('NetMHC')
methods.insert(0, 'NetMHC')

#['NetMHC', 'puffin', 'mhcnuggets', 'acme', 'mhcseqnet', 'mhcattnnet']

per_allele_per_method_predictions = dict()

def less_is_good(column):
    if column.find('affinity') >= 0 :
        return True
    if column.find('percentile_rank') >=0:
        return True
    if column in ['epistemic_var', 'aleatoric_var', 'HMMB']:
        return True
    return False

for allele_name in ALLELES:
    print(f"{allele_name}:", end=" ")
    per_allele_per_method_predictions[allele_name] = dict()
    for method in methods:
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
        prediction_df = pd.read_csv(f"{PREDICTION_PATH}{method}/{method}_predictions_allele_{target_allele_name}.csv", sep=';')

        #add diff 

        print(f"{method}", end=" ")
        for score_column in per_method_columns[method]:
            result_score_column = f"{method}_{score_column}" if not (method == 'NetMHC') else score_column
            wt_scores = prediction_df[result_score_column + '_wt'].values
            mt_scores = prediction_df[result_score_column + '_mt'].values
            wt_peptide =  prediction_df['peptide_wt'].values
            # for these columns mt shoud be less than wt. In this case wt/mt will be > 1 and this is good. 
            if less_is_good(score_column):
                advantage_ratio = np.array([wt_sc/(mt_sc + 0.0000000001) if not peptide == np.nan else 2  for wt_sc,mt_sc, peptide in zip(wt_scores,mt_scores,wt_peptide)])
                advantage_delta = np.array([wt_sc - mt_sc if not peptide == np.nan else 2  for wt_sc,mt_sc, peptide in zip(wt_scores,mt_scores,wt_peptide)])

        # for these columns mt shoud be greater than wt. In this case mt/wt will be > 1 and this is good.
            else:
                advantage_ratio = np.array([mt_sc/(wt_sc + 0.0000000001) if not peptide == np.nan else 2  for wt_sc,mt_sc, peptide in zip(wt_scores,mt_scores,wt_peptide)])
                advantage_delta = np.array([mt_sc - wt_sc if not peptide == np.nan else 2  for wt_sc,mt_sc, peptide in zip(wt_scores,mt_scores,wt_peptide)])
            prediction_df[f"{result_score_column}_advantage_ratio"] = advantage_ratio
            prediction_df[f"{result_score_column}_advantage_delta"] = advantage_delta



        per_allele_per_method_predictions[allele_name][method] = prediction_df
    print()


H-2-Dd: NetMHC ghmm mhcflurry mhcnuggets puffin 
H-2-Ld: NetMHC ghmm mhcflurry mhcnuggets puffin 
H-2-Kd: NetMHC ghmm mhcflurry mhcnuggets puffin 


In [115]:
per_allele_per_method_predictions[ALLELES[1]]['NetMHC']

,source_sequence_name,allele,peptide_mt,length,netMHC-3.4_score_mt,netMHC-3.4_affinity_mt,netMHC-3.4_percentile_rank_mt,netMHC-4.0_score_mt,netMHC-4.0_affinity_mt,netMHC-4.0_percentile_rank_mt,...,netMHCIIpan-4.0_score_ba_advantage_ratio,netMHCIIpan-4.0_score_ba_advantage_delta,netMHCIIpan-4.0_score_el_advantage_ratio,netMHCIIpan-4.0_score_el_advantage_delta,netMHCIIpan-4.0_affinity_advantage_ratio,netMHCIIpan-4.0_affinity_advantage_delta,netMHCIIpan-4.0_percentile_rank_ba_advantage_ratio,netMHCIIpan-4.0_percentile_rank_ba_advantage_delta,netMHCIIpan-4.0_percentile_rank_el_advantage_ratio,netMHCIIpan-4.0_percentile_rank_el_advantage_delta
0,seq0,H-2-Ld,FQDSLINQMTQS,12,0.008,46087.0,NaN,0.013,43395.20,39.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,seq1,H-2-Ld,QDSLINQMTQSK,12,0.006,46920.0,NaN,0.008,45679.31,60.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,seq2,H-2-Ld,DSLINQMTQSKL,12,0.013,43484.0,NaN,0.038,33267.20,13.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,seq3,H-2-Ld,SLINQMTQSKLS,12,0.007,46203.0,NaN,0.041,31934.06,12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,seq4,H-2-Ld,LINQMTQSKLSV,12,0.018,41181.0,NaN,0.046,30478.39,11.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2728,seq2728,H-2-Ld,ARYMLQNPHIIA,12,0.007,46143.0,NaN,0.035,34140.90,14.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2729,seq2729,H-2-Ld,RYMLQNPHIIAT,12,0.008,45815.0,NaN,0.030,36069.96,17.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2730,seq2730,H-2-Ld,YMLQNPHIIATK,12,0.008,45740.0,NaN,0.025,37994.82,21.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2731,seq2731,H-2-Ld,MLQNPHIIATKT,12,0.012,44124.0,NaN,0.091,18697.03,4.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 2. Merge predictions

In [116]:
per_allele_predictions = dict()
for allele_name in ALLELES:
    list_for_allele = list(per_allele_per_method_predictions[allele_name].values())

    result_df = list_for_allele[0].copy()
    for df_to_merge in list_for_allele[1:]:
        result_df = result_df.merge(df_to_merge, on=['source_sequence_name', 'peptide_mt', 'length', 'peptide_wt', 'allele'], how='left')
    result_df = result_df[['source_sequence_name','allele', 'length'] + [column for column in result_df.columns if column.endswith('mt')] 
                          + [column for column in result_df.columns if column.endswith('_advantage_ratio')]
                          + [column for column in result_df.columns if column.endswith('_advantage_delta')]
                          + [column for column in result_df.columns if column.endswith('wt')]]
    per_allele_predictions[allele_name] = result_df


In [117]:
all_methods = list(set([item.split('_')[0] for item in result_df.columns if  item.split('_')[0] not in ['allele', 'peptide', 'length', 'source']]))
all_methods

['netMHC-4.0',
 'ghmm',
 'netMHC-3.4',
 'netMHCIIpan-4.0',
 'puffin',
 'netMHCpan-4.1',
 'mhcflurry',
 'mhcnuggets']

In [118]:
unsupported_alleles = {method : set() for method in all_methods}

def add_unsupported_allele(method, alleles):
    methods_set = set(all_methods)
    if (method in methods_set):
        unsupported_alleles[method].update(alleles)

for allele_name in ALLELES:
    if defineClass(allele_name) == 'II':
        add_unsupported_allele('netMHCpan-4.1', [allele_name])
        add_unsupported_allele('netMHC-3.4', [allele_name])
        add_unsupported_allele('netMHC-4.0', [allele_name])
        add_unsupported_allele('acme', [allele_name])
        add_unsupported_allele('mhcflurry', [allele_name])
    else:
        add_unsupported_allele('netMHCIIpan-4.0', [allele_name])
        
    if not defineOrganism(allele_name) == 'Human':
        add_unsupported_allele('mhcseqnet', [allele_name])
        add_unsupported_allele('mhcattnnet', [allele_name])
        add_unsupported_allele('acme', [allele_name])

#special cases
add_unsupported_allele('acme',["HLA-C*04:01", "HLA-C*03:04"])
add_unsupported_allele('puffin',["HLA-C*04:01"])


def drop_unsupported_columns(per_allele_df):
    per_allele_all_methods = {allele_name : set() for allele_name in per_allele_df.keys()}

    for allele_name, df in per_allele_df.items():
        for method in all_methods:
            if allele_name in unsupported_alleles[method]:
                columns_to_drop = [col.strip() for col in df.columns if col.startswith(method)] + ['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt']
                df = df.drop(columns=columns_to_drop, errors='ignore')
            else:
                per_allele_all_methods[allele_name].add(method)
            per_allele_df[allele_name] = df
        
            #df = df.drop(columns=['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt'])
    return per_allele_df, per_allele_all_methods



# Add source info

In [119]:

source_total_columns = ['VAR_ID', 'mutant_id', 'transcript_id', 'peptide_offset',
                        'A20_RGT', 'A20_RDP', 'A20_RAD_REF',
                        'A20_RAD_ALT', 'A20_RAF_ALT', 'A20_RPKM_mut',  'source_sequence_name', 
                        'peptide_wt', 'peptide_mt' , 'allele',  'length'
                       ]
source_total_df = source_total_df[source_total_columns]
source_total_df

,VAR_ID,mutant_id,transcript_id,peptide_offset,A20_RGT,A20_RDP,A20_RAD_REF,A20_RAD_ALT,A20_RAF_ALT,A20_RPKM_mut,source_sequence_name,peptide_wt,peptide_mt,allele,length
0,1_37366236_T/G,Inpp4a-I115S,ENSMUST00000137266,103,T/G,87,42,45,0.517241,0.437313,seq0,FQDSLINQMTQI,FQDSLINQMTQS,H-2-Kd,12
1,1_37366236_T/G,Inpp4a-I115S,ENSMUST00000137266,104,T/G,87,42,45,0.517241,0.437313,seq1,QDSLINQMTQIK,QDSLINQMTQSK,H-2-Kd,12
2,1_37366236_T/G,Inpp4a-I115S,ENSMUST00000137266,105,T/G,87,42,45,0.517241,0.437313,seq2,DSLINQMTQIKL,DSLINQMTQSKL,H-2-Kd,12
3,1_37366236_T/G,Inpp4a-I115S,ENSMUST00000137266,106,T/G,87,42,45,0.517241,0.437313,seq3,SLINQMTQIKLS,SLINQMTQSKLS,H-2-Kd,12
4,1_37366236_T/G,Inpp4a-I115S,ENSMUST00000137266,107,T/G,87,42,45,0.517241,0.437313,seq4,LINQMTQIKLSV,LINQMTQSKLSV,H-2-Kd,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8194,X_162769854_C/T,Rbbp7-P133L,ENSMUST00000033720,128,C/T,3266,2,3264,0.999388,52.215983,seq2728,ARYMPQNPHIIA,ARYMLQNPHIIA,H-2-Dd,12
8195,X_162769854_C/T,Rbbp7-P133L,ENSMUST00000033720,129,C/T,3266,2,3264,0.999388,52.215983,seq2729,RYMPQNPHIIAT,RYMLQNPHIIAT,H-2-Dd,12
8196,X_162769854_C/T,Rbbp7-P133L,ENSMUST00000033720,130,C/T,3266,2,3264,0.999388,52.215983,seq2730,YMPQNPHIIATK,YMLQNPHIIATK,H-2-Dd,12
8197,X_162769854_C/T,Rbbp7-P133L,ENSMUST00000033720,131,C/T,3266,2,3264,0.999388,52.215983,seq2731,MPQNPHIIATKT,MLQNPHIIATKT,H-2-Dd,12


In [120]:
per_method_main_scores = {
    'netMHC-3.4': 'affinity',
    'netMHC-4.0': 'affinity',
    'netMHCpan-4.1': 'score_el',
    'netMHCIIpan-4.0' : 'score_el',
    'mhcflurry': 'score_el',
    'mhcnuggets': 'affinity',
    'acme':'score',
    'puffin':'score',
    'mhcseqnet':'score',
    'mhcattnnet': 'score',
    'ghmm': 'HMMB',
}

per_method_main_percentiles = {
    'netMHC-4.0':'percentile_rank' ,
    'netMHCpan-4.1': 'percentile_rank_el',
    'netMHCIIpan-4.0' : 'percentile_rank_el',
    'mhcflurry':'percentile_rank_el' ,
    'mhcnuggets': 'percentile_rank',
    #'acme':'score',
    'puffin': 'percentile_rank',
    #'mhcseqnet': 'score',
    #'mhcattnnet': 'score',
    'ghmm': 'percentile_rank'
}

def reorder_columns(df, source_total_columns, source_id_columns):
    general_info = ['mutant_id', 'allele', 'length', 'median_percentile','mean_percentile',
                    'median_percentile_delta', 'mean_percentile_delta', 'median_score_ratio', 'A20_RPKM_mut']



    source_columns = [source_column for source_column in source_total_columns
                          if source_column not in general_info
                          and source_column not in source_id_columns
                          and source_column not in ['peptide_wt', 'peptide_mt']]


    percentile_columns_mt = [column for column in df.columns if column.find('percentile')>=0 and column.endswith('mt')]
    percentile_columns_wt = [column for column in df.columns if column.find('percentile')>=0 and column.endswith('wt')]
    percentile_columns = percentile_columns_mt + percentile_columns_wt
    not_percentile_columns = [column for column in df.columns 
                              if column not in percentile_columns 
                              and  column not in general_info 
                              and column not in ['peptide_wt', 'peptide_mt'] 
                              and column not in source_columns]

    columns_ordered = list()
    columns_ordered.extend(general_info)
    columns_ordered.append('peptide_mt')
    columns_ordered.extend(percentile_columns_mt)

    columns_ordered.append('peptide_wt')
    columns_ordered.extend(percentile_columns_wt)
    columns_ordered.extend(source_columns)
    columns_ordered.extend(not_percentile_columns)
    columns_ordered.append(RESULT_COLUMNS['id'])

            #df = df.drop(columns=['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt'])
    return df[columns_ordered]

def add_avg_percentile_for_selected(per_allele_predictions, selected_methods, per_allele_methods):
    per_allele_result = dict()
    for allele_name, df in per_allele_predictions.items():
        selected_percentiles = [method + "_" + per_method_main_percentiles[method] for method in selected_methods if method in per_allele_methods[allele_name]]
        selected_percentiles_columns = [percentile + "_mt" for percentile in selected_percentiles]
        selected_percentiles_advantage_delta = [percentile + "_advantage_delta" for percentile in selected_percentiles]

        selected_main_scores = [method + "_" + per_method_main_scores[method]  for method in selected_methods if method in per_allele_methods[allele_name]]
        selected_main_scores_advantage_ratio = [score  + "_advantage_ratio" for score in selected_main_scores]

        df['median_percentile'] = df[selected_percentiles_columns].median(axis=1)
        df['mean_percentile'] = df[selected_percentiles_columns].mean(axis=1)
        df['median_percentile_delta'] = df[selected_percentiles_advantage_delta].median(axis=1)
        df['mean_percentile_delta'] = df[selected_percentiles_advantage_delta].mean(axis=1)

        df['median_score_ratio'] = df[selected_main_scores_advantage_ratio].median(axis=1)

        per_allele_result[allele_name] = df
    return per_allele_result




per_allele_predictions, per_allele_methods = drop_unsupported_columns(per_allele_predictions)
per_allele_predictions = add_avg_percentile_for_selected(per_allele_predictions,
                                                         selected_methods = ['netMHCIIpan-4.0', 'netMHCpan-4.1', 'mhcflurry','mhcnuggets','puffin'],
                                                         per_allele_methods=per_allele_methods)




class_I_dfs = list()
class_II_dfs = list()
class_I_total_df = None
class_II_total_df = None
for allele_name in ALLELES:
    df_allele_source = source_total_df.loc[source_total_df['allele'] == allele_name].copy()
    df_predictions = per_allele_predictions[allele_name]
    df_result = df_predictions.merge(df_allele_source, on=['source_sequence_name', 'peptide_mt', 'length', 'peptide_wt', 'allele'])
    df_result = reorder_columns(df_result, source_total_columns, source_id_columns=['mutand_id'])
    per_allele_predictions[allele_name] = df_result
    if defineClass(allele_name) == 'I':
        class_I_dfs.append(df_result)
    elif defineClass(allele_name) == 'II':
        class_II_dfs.append(df_result)

if class_I_dfs:
    class_I_total_df = pd.concat(class_I_dfs)
if class_II_dfs:
    class_II_total_df = pd.concat(class_II_dfs)




In [121]:
per_allele_predictions[ALLELES[0]].columns

Index(['mutant_id', 'allele', 'length', 'median_percentile', 'mean_percentile',
       'median_percentile_delta', 'mean_percentile_delta',
       'median_score_ratio', 'A20_RPKM_mut', 'peptide_mt',
       ...
       'mhcflurry_affinity_wt', 'mhcflurry_processing_score_wt',
       'mhcflurry_score_el_wt', 'mhcnuggets_score_wt',
       'mhcnuggets_affinity_wt', 'puffin_score_wt', 'puffin_epistemic_var_wt',
       'puffin_aleatoric_var_wt', 'puffin_binding_likelihood_wt',
       'source_sequence_name'],
      dtype='object', length=121)

In [122]:
per_allele_methods

{'H-2-Dd': {'ghmm',
  'mhcflurry',
  'mhcnuggets',
  'netMHC-3.4',
  'netMHC-4.0',
  'netMHCpan-4.1',
  'puffin'},
 'H-2-Ld': {'ghmm',
  'mhcflurry',
  'mhcnuggets',
  'netMHC-3.4',
  'netMHC-4.0',
  'netMHCpan-4.1',
  'puffin'},
 'H-2-Kd': {'ghmm',
  'mhcflurry',
  'mhcnuggets',
  'netMHC-3.4',
  'netMHC-4.0',
  'netMHCpan-4.1',
  'puffin'}}

In [123]:
sum([len(df) for df in per_allele_mt_df.values()])

8199

In [124]:
len(source_total_df)

8199

## Save total pred to file

In [125]:
# for decoy_key in DECOY_PREFIXES:
for allele_name in ALLELES:
    total_df = per_allele_predictions[allele_name]
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df.to_csv("{}total_predictions_allele_{}.csv".format(PREDICTION_PATH,target_allele_name),sep=";", index=False)

In [126]:
if class_I_total_df is not None:
    class_I_total_df.to_csv("{}total_predictions_class_I.csv".format(PREDICTION_PATH),sep=";", index=False)
if class_II_total_df is not None:
    class_II_total_df.to_csv("{}total_predictions_class_II.csv".format(PREDICTION_PATH),sep=";", index=False)

In [127]:
class_I_total_df

,mutant_id,allele,length,median_percentile,mean_percentile,median_percentile_delta,mean_percentile_delta,median_score_ratio,A20_RPKM_mut,peptide_mt,...,mhcflurry_affinity_wt,mhcflurry_processing_score_wt,mhcflurry_score_el_wt,mhcnuggets_score_wt,mhcnuggets_affinity_wt,puffin_score_wt,puffin_epistemic_var_wt,puffin_aleatoric_var_wt,puffin_binding_likelihood_wt,source_sequence_name
0,Inpp4a-I115S,H-2-Dd,12,87.175250,77.338713,-47.728340,-46.936045,0.517049,0.437313,FQDSLINQMTQS,...,24258.577173,0.074636,0.005382,0.044855,30774.87,0.073275,0.000687,0.009089,0.005547,seq0
1,Inpp4a-I115S,H-2-Dd,12,65.612337,67.029668,-4.178000,-3.097938,0.873299,0.437313,QDSLINQMTQSK,...,26663.735087,0.051928,0.004535,0.030538,35931.37,0.026802,0.000177,0.001888,0.000001,seq1
2,Inpp4a-I115S,H-2-Dd,12,29.918024,35.755137,-7.824905,-15.818577,0.784046,0.437313,DSLINQMTQSKL,...,27587.591350,0.256764,0.009015,0.073110,22668.71,0.062145,0.000574,0.006995,0.001857,seq2
3,Inpp4a-I115S,H-2-Dd,12,64.809750,66.461713,-2.436000,-11.087438,0.866748,0.437313,SLINQMTQSKLS,...,29934.441205,0.030164,0.003755,0.056855,27027.63,0.026159,0.000142,0.001815,0.000001,seq3
4,Inpp4a-I115S,H-2-Dd,12,75.060750,67.716838,-0.299500,0.642688,0.982556,0.437313,LINQMTQSKLSV,...,30377.298389,0.048251,0.003946,0.025805,37819.20,0.045969,0.000296,0.004368,0.000391,seq4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2728,Rbbp7-P133L,H-2-Kd,12,22.274875,32.384363,-7.542299,-4.481899,0.736199,52.215983,ARYMLQNPHIIA,...,25964.231219,0.548389,0.026362,0.043346,31281.47,0.123273,0.000564,0.021754,0.051124,seq2728
2729,Rbbp7-P133L,H-2-Kd,12,5.041709,5.205855,-1.812041,-1.541270,0.635363,52.215983,RYMLQNPHIIAT,...,9847.543613,0.733759,0.117999,0.247849,3422.44,0.227844,0.002134,0.044999,0.183067,seq2729
2730,Rbbp7-P133L,H-2-Kd,12,12.700921,11.693961,2.061750,7.822740,1.090890,52.215983,YMLQNPHIIATK,...,23630.547348,0.624764,0.037420,0.305889,1826.45,0.108734,0.000509,0.018197,0.035181,seq2730
2731,Rbbp7-P133L,H-2-Kd,12,62.079125,66.550776,-19.934250,-27.592529,0.490383,52.215983,MLQNPHIIATKT,...,30386.307480,0.373676,0.012370,0.160688,8788.23,0.098523,0.000624,0.015867,0.026575,seq2731


In [128]:
per_allele_predictions[allele_name]

,mutant_id,allele,length,median_percentile,mean_percentile,median_percentile_delta,mean_percentile_delta,median_score_ratio,A20_RPKM_mut,peptide_mt,...,mhcflurry_affinity_wt,mhcflurry_processing_score_wt,mhcflurry_score_el_wt,mhcnuggets_score_wt,mhcnuggets_affinity_wt,puffin_score_wt,puffin_epistemic_var_wt,puffin_aleatoric_var_wt,puffin_binding_likelihood_wt,source_sequence_name
0,Inpp4a-I115S,H-2-Kd,12,57.951625,60.374526,-39.033750,-39.015857,0.533198,0.437313,FQDSLINQMTQS,...,24192.786458,0.074636,0.005396,0.082418,20496.91,0.304929,0.003825,0.053741,0.284097,seq0
1,Inpp4a-I115S,H-2-Kd,12,74.562250,73.748543,-5.033000,-8.978000,0.880783,0.437313,QDSLINQMTQSK,...,29407.095002,0.051928,0.004125,0.049930,29130.61,0.093745,0.000498,0.014288,0.021684,seq1
2,Inpp4a-I115S,H-2-Kd,12,26.059875,25.914043,0.194500,-0.561321,0.935073,0.437313,DSLINQMTQSKL,...,29610.420439,0.256764,0.008421,0.264251,2865.92,0.120302,0.000271,0.020790,0.048726,seq2
3,Inpp4a-I115S,H-2-Kd,12,51.714125,55.817526,-0.590500,-3.030500,0.963372,0.437313,SLINQMTQSKLS,...,30732.139345,0.030164,0.003661,0.195751,6013.71,0.108760,0.000414,0.017987,0.035546,seq3
4,Inpp4a-I115S,H-2-Kd,12,34.497625,43.583651,-0.768000,-0.226875,0.965255,0.437313,LINQMTQSKLSV,...,31526.439995,0.048251,0.003807,0.270458,2679.77,0.140153,0.000556,0.025915,0.071209,seq4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2728,Rbbp7-P133L,H-2-Kd,12,22.274875,32.384363,-7.542299,-4.481899,0.736199,52.215983,ARYMLQNPHIIA,...,25964.231219,0.548389,0.026362,0.043346,31281.47,0.123273,0.000564,0.021754,0.051124,seq2728
2729,Rbbp7-P133L,H-2-Kd,12,5.041709,5.205855,-1.812041,-1.541270,0.635363,52.215983,RYMLQNPHIIAT,...,9847.543613,0.733759,0.117999,0.247849,3422.44,0.227844,0.002134,0.044999,0.183067,seq2729
2730,Rbbp7-P133L,H-2-Kd,12,12.700921,11.693961,2.061750,7.822740,1.090890,52.215983,YMLQNPHIIATK,...,23630.547348,0.624764,0.037420,0.305889,1826.45,0.108734,0.000509,0.018197,0.035181,seq2730
2731,Rbbp7-P133L,H-2-Kd,12,62.079125,66.550776,-19.934250,-27.592529,0.490383,52.215983,MLQNPHIIATKT,...,30386.307480,0.373676,0.012370,0.160688,8788.23,0.098523,0.000624,0.015867,0.026575,seq2731
